In [ ]:
from ultralytics import YOLO

: 

In [21]:
from ultralytics import YOLO
import cv2
import numpy as np

# =========================
# 경로 설정
# =========================
MODEL_PATH = r"C:\Users\nva_kist\Desktop\minsun\KIST\Tomato_detect\weights1.pt"
RGB_VIDEO_PATH = r"C:\Users\nva_kist\Desktop\minsun\KIST\Tomato_detect\rgb_5l3.mp4"      # 원본 RGB 영상
DEPTH_VIDEO_PATH = r"C:\Users\nva_kist\Desktop\minsun\KIST\Tomato_detect\depth_zed.mp4"  # depth 영상
OUTPUT_PATH = r"C:\Users\nva_kist\Desktop\minsun\KIST\Tomato_detect\5l1_depth.mp4"

# =========================
# 파라미터
# =========================
CONF_THRES = 0.30
MIN_BOX_AREA = 2000
DEPTH_THRESH = 140   # 예시값: 이 값은 depth 영상 범위 보고 조정 필요

model = YOLO(MODEL_PATH)

cap_rgb = cv2.VideoCapture(RGB_VIDEO_PATH)
cap_depth = cv2.VideoCapture(DEPTH_VIDEO_PATH)

if not cap_rgb.isOpened():
    raise FileNotFoundError(f"RGB 영상을 열 수 없습니다: {RGB_VIDEO_PATH}")
if not cap_depth.isOpened():
    raise FileNotFoundError(f"Depth 영상을 열 수 없습니다: {DEPTH_VIDEO_PATH}")

fps = cap_rgb.get(cv2.CAP_PROP_FPS)
width = int(cap_rgb.get(cv2.CAP_PROP_FRAME_WIDTH))
height = int(cap_rgb.get(cv2.CAP_PROP_FRAME_HEIGHT))

fourcc = cv2.VideoWriter_fourcc(*"mp4v")
out = cv2.VideoWriter(OUTPUT_PATH, fourcc, fps, (width, height))

names = model.names

while True:
    ret_rgb, frame_rgb = cap_rgb.read()
    ret_depth, frame_depth = cap_depth.read()

    if not ret_rgb or not ret_depth:
        break

    # depth 영상 해상도 맞추기
    frame_depth = cv2.resize(frame_depth, (frame_rgb.shape[1], frame_rgb.shape[0]))

    # depth 영상이 컬러맵 형태라면 gray로 변환
    if len(frame_depth.shape) == 3:
        depth_gray = cv2.cvtColor(frame_depth, cv2.COLOR_BGR2GRAY)
    else:
        depth_gray = frame_depth.copy()

    results = model(frame_rgb, conf=CONF_THRES, verbose=False)
    r = results[0]

    if r.boxes is not None:
        for box in r.boxes:
            x1, y1, x2, y2 = map(int, box.xyxy[0].tolist())
            conf = float(box.conf[0])
            cls = int(box.cls[0])

            # bbox area 기준 작은 것 제거
            area = (x2 - x1) * (y2 - y1)
            if area < MIN_BOX_AREA:
                continue

            # bbox 내부 depth 추출
            roi = depth_gray[y1:y2, x1:x2]
            if roi.size == 0:
                continue

            # 너무 가장자리 값 영향 줄이려고 가운데 60%만 사용
            h, w = roi.shape
            y_start = int(h * 0.2)
            y_end = int(h * 0.8)
            x_start = int(w * 0.2)
            x_end = int(w * 0.8)
            roi_center = roi[y_start:y_end, x_start:x_end]

            if roi_center.size == 0:
                continue

            depth_median = np.median(roi_center)

            # depth 기준 필터
            # depth 영상에서 "밝을수록 가까움"인지 "어두울수록 가까움"인지 꼭 확인 필요
            # 아래는 예시로 depth 값이 작을수록 가까운 경우
            if depth_median > DEPTH_THRESH:
                continue

            label = f"{names[cls]} {conf:.2f} d={depth_median:.1f}"
            cv2.rectangle(frame_rgb, (x1, y1), (x2, y2), (0, 255, 0), 2)
            cv2.putText(frame_rgb, label, (x1, max(y1 - 8, 0)),
                        cv2.FONT_HERSHEY_SIMPLEX, 0.6, (0, 255, 0), 2)

    out.write(frame_rgb)
    cv2.imshow("depth filtered", frame_rgb)

    if cv2.waitKey(1) & 0xFF == ord("q"):
        break

cap_rgb.release()
cap_depth.release()
out.release()
cv2.destroyAllWindows()
print(f"cls={names[cls]}, area={area}, depth_median={depth_median:.1f}")

print(f"저장 완료: {OUTPUT_PATH}")

cls=unripe tomato, area=3894, depth_median=20.0
저장 완료: C:\Users\nva_kist\Desktop\minsun\KIST\Tomato_detect\5l1_depth.mp4


In [7]:
!C:\Users\nva_kist\AppData\Local\Programs\Python\Python311\python.exe -m pip install lap

   ---------------------------------------- 0.0/1.5 MB ? eta -:--:--
   ---------------------------------------- 1.5/1.5 MB 39.2 MB/s  0:00:00


In [15]:
# yolo only (track id 제거)
from ultralytics import YOLO
import cv2
import numpy as np

# =========================
# 경로 설정
# =========================
MODEL_PATH = r"C:\Users\nva_kist\Desktop\minsun\KIST\Tomato_detect\runs\detect\train3\weights\best.pt"
RGB_VIDEO_PATH = r"C:\Users\nva_kist\Desktop\minsun\KIST\Tomato_detect\Video\rgb_5l3.mp4"
DEPTH_VIDEO_PATH = r"C:\Users\nva_kist\Desktop\minsun\KIST\Tomato_detect\Video\depth_zed.mp4"
OUTPUT_PATH = r"C:\Users\nva_kist\Desktop\minsun\KIST\Tomato_detect\Video\5l1_depth_public.mp4"

# =========================
# 파라미터
# =========================
CONF_THRES = 0.30
MIN_BOX_AREA = 2000
DEPTH_THRESH = 140
IOU_DUP_THRESH = 0.6   # 겹침이 이 이상이면 중복으로 간주

model = YOLO(MODEL_PATH)

cap_rgb = cv2.VideoCapture(RGB_VIDEO_PATH)
cap_depth = cv2.VideoCapture(DEPTH_VIDEO_PATH)

if not cap_rgb.isOpened():
    raise FileNotFoundError(f"RGB 영상을 열 수 없습니다: {RGB_VIDEO_PATH}")
if not cap_depth.isOpened():
    raise FileNotFoundError(f"Depth 영상을 열 수 없습니다: {DEPTH_VIDEO_PATH}")

fps = cap_rgb.get(cv2.CAP_PROP_FPS)
width = int(cap_rgb.get(cv2.CAP_PROP_FRAME_WIDTH))
height = int(cap_rgb.get(cv2.CAP_PROP_FRAME_HEIGHT))

fourcc = cv2.VideoWriter_fourcc(*"mp4v")
out = cv2.VideoWriter(OUTPUT_PATH, fourcc, fps, (width, height))

names = model.names


def compute_iou(box1, box2):
    x1 = max(box1[0], box2[0])
    y1 = max(box1[1], box2[1])
    x2 = min(box1[2], box2[2])
    y2 = min(box1[3], box2[3])

    inter_w = max(0, x2 - x1)
    inter_h = max(0, y2 - y1)
    inter = inter_w * inter_h

    area1 = max(0, box1[2] - box1[0]) * max(0, box1[3] - box1[1])
    area2 = max(0, box2[2] - box2[0]) * max(0, box2[3] - box2[1])
    union = area1 + area2 - inter

    if union == 0:
        return 0.0
    return inter / union


def remove_duplicate_boxes(candidates, iou_thresh=0.6):
    candidates = sorted(candidates, key=lambda x: x["conf"], reverse=True)
    kept = []

    for cand in candidates:
        is_duplicate = False
        for kept_box in kept:
            iou = compute_iou(cand["xyxy"], kept_box["xyxy"])
            if iou >= iou_thresh:
                is_duplicate = True
                break
        if not is_duplicate:
            kept.append(cand)

    return kept


while True:
    ret_rgb, frame_rgb = cap_rgb.read()
    ret_depth, frame_depth = cap_depth.read()

    if not ret_rgb or not ret_depth:
        break

    frame_depth = cv2.resize(frame_depth, (frame_rgb.shape[1], frame_rgb.shape[0]))

    if len(frame_depth.shape) == 3:
        depth_gray = cv2.cvtColor(frame_depth, cv2.COLOR_BGR2GRAY)
    else:
        depth_gray = frame_depth.copy()

    # tracking 대신 detection만 수행
    results = model(
        frame_rgb,
        conf=CONF_THRES,
        verbose=False
    )
    r = results[0]

    candidates = []

    if r.boxes is not None:
        for box in r.boxes:
            x1, y1, x2, y2 = map(int, box.xyxy[0].tolist())
            conf = float(box.conf[0])
            cls = int(box.cls[0])

            area = (x2 - x1) * (y2 - y1)
            if area < MIN_BOX_AREA:
                continue

            roi = depth_gray[y1:y2, x1:x2]
            if roi.size == 0:
                continue

            h, w = roi.shape
            y_start = int(h * 0.2)
            y_end = int(h * 0.8)
            x_start = int(w * 0.2)
            x_end = int(w * 0.8)
            roi_center = roi[y_start:y_end, x_start:x_end]

            if roi_center.size == 0:
                continue

            depth_median = np.median(roi_center)

            if depth_median > DEPTH_THRESH:
                continue

            candidates.append({
                "xyxy": [x1, y1, x2, y2],
                "conf": conf,
                "cls": cls,
                "depth_median": depth_median,
                "area": area
            })

    # =========================
    # 중복 박스 제거
    # =========================
    final_boxes = remove_duplicate_boxes(candidates, iou_thresh=IOU_DUP_THRESH)

    for item in final_boxes:
        x1, y1, x2, y2 = item["xyxy"]
        conf = item["conf"]
        cls = item["cls"]
        depth_median = item["depth_median"]

        class_name = names[cls].lower()

        if class_name == "ripe":
            color = (0, 0, 255)   # 빨강
        elif class_name == "unripe":
            color = (0, 255, 0)   # 초록
        else:
            color = (255, 255, 255)

        label = f"{names[cls]} {conf:.2f} d={depth_median:.1f}"
        cv2.rectangle(frame_rgb, (x1, y1), (x2, y2), color, 2)
        cv2.putText(frame_rgb, label, (x1, max(y1 - 8, 0)),
                    cv2.FONT_HERSHEY_SIMPLEX, 0.6, color, 2)

    out.write(frame_rgb)
    cv2.imshow("depth filtered", frame_rgb)

    if cv2.waitKey(1) & 0xFF == ord("q"):
        break

cap_rgb.release()
cap_depth.release()
out.release()
cv2.destroyAllWindows()

print(f"저장 완료: {OUTPUT_PATH}")

저장 완료: C:\Users\nva_kist\Desktop\minsun\KIST\Tomato_detect\Video\5l1_depth_public.mp4


#### Dataset 나누기

In [31]:
import os
import json
import random
import shutil
from pathlib import Path
from collections import defaultdict

# =========================
# 사용자 설정
# =========================
ROOT_DIR = r"C:\Users\nva_kist\Desktop\minsun\KIST\Tomato_detect\anotationcoco_all\train"
COCO_JSON = os.path.join(ROOT_DIR, "_annotations.coco.json")

OUTPUT_DIR = r"C:\Users\nva_kist\Desktop\minsun\KIST\Tomato_detect\yolo_dataset"
VAL_RATIO = 0.2
SEED = 42

# 사용할 클래스만 선택
# COCO categories:
# 0: tomatoes
# 1: ripe
# 2: unripe
USE_CATEGORY_IDS = [1, 2]   # ripe, unripe만 사용
YOLO_CLASS_MAP = {
    1: 0,   # ripe -> class 0
    2: 1    # unripe -> class 1
}
YOLO_CLASS_NAMES = ["ripe", "unripe"]

# =========================
# 폴더 생성
# =========================
train_img_dir = os.path.join(OUTPUT_DIR, "images", "train")
val_img_dir = os.path.join(OUTPUT_DIR, "images", "val")
train_lbl_dir = os.path.join(OUTPUT_DIR, "labels", "train")
val_lbl_dir = os.path.join(OUTPUT_DIR, "labels", "val")

for d in [train_img_dir, val_img_dir, train_lbl_dir, val_lbl_dir]:
    os.makedirs(d, exist_ok=True)

# =========================
# COCO 로드
# =========================
with open(COCO_JSON, "r", encoding="utf-8") as f:
    coco = json.load(f)

images = coco["images"]
annotations = coco["annotations"]
categories = coco["categories"]

print("총 이미지 수:", len(images))
print("총 annotation 수:", len(annotations))
print("categories:", categories)

# image_id -> image_info
image_dict = {img["id"]: img for img in images}

# image_id -> annotations
ann_dict = defaultdict(list)
for ann in annotations:
    if ann["category_id"] in USE_CATEGORY_IDS:
        ann_dict[ann["image_id"]].append(ann)

# annotation이 하나라도 있는 이미지만 사용하고 싶으면 아래 사용
valid_images = [img for img in images if len(ann_dict[img["id"]]) > 0]

# annotation 없는 이미지까지 포함하려면 아래로 바꾸기
# valid_images = images

print("사용할 이미지 수:", len(valid_images))

# =========================
# train / val split
# =========================
random.seed(SEED)
random.shuffle(valid_images)

n_val = int(len(valid_images) * VAL_RATIO)
val_images = valid_images[:n_val]
train_images = valid_images[n_val:]

print(f"train: {len(train_images)}장")
print(f"val: {len(val_images)}장")

# =========================
# bbox 변환 함수
# COCO bbox = [x_min, y_min, width, height]
# YOLO bbox = [class_id, x_center, y_center, width, height] (0~1 정규화)
# =========================
def coco_bbox_to_yolo(bbox, img_w, img_h):
    x_min, y_min, box_w, box_h = map(float, bbox)
    img_w = float(img_w)
    img_h = float(img_h)

    x_center = x_min + box_w / 2.0
    y_center = y_min + box_h / 2.0

    x_center /= img_w
    y_center /= img_h
    box_w /= img_w
    box_h /= img_h

    return x_center, y_center, box_w, box_h
# =========================
# split 저장 함수
# =========================
def save_split(split_images, img_out_dir, lbl_out_dir):
    for img_info in split_images:
        img_id = img_info["id"]
        file_name = img_info["file_name"]
        img_w = img_info["width"]
        img_h = img_info["height"]

        src_img_path = os.path.join(ROOT_DIR, file_name)
        dst_img_path = os.path.join(img_out_dir, file_name)

        if not os.path.exists(src_img_path):
            print(f"[경고] 이미지 없음: {src_img_path}")
            continue

        # 이미지 복사
        shutil.copy2(src_img_path, dst_img_path)

        # 라벨 txt 저장
        txt_name = Path(file_name).stem + ".txt"
        txt_path = os.path.join(lbl_out_dir, txt_name)

        yolo_lines = []
        for ann in ann_dict[img_id]:
            cat_id = ann["category_id"]

            if cat_id not in YOLO_CLASS_MAP:
                continue

            yolo_cls = YOLO_CLASS_MAP[cat_id]
            x_center, y_center, box_w, box_h = coco_bbox_to_yolo(
                ann["bbox"], img_w, img_h
            )

            line = f"{yolo_cls} {x_center:.6f} {y_center:.6f} {box_w:.6f} {box_h:.6f}"
            yolo_lines.append(line)

        with open(txt_path, "w", encoding="utf-8") as f:
            f.write("\n".join(yolo_lines))

# =========================
# 저장 실행
# =========================
save_split(train_images, train_img_dir, train_lbl_dir)
save_split(val_images, val_img_dir, val_lbl_dir)

# =========================
# data.yaml 생성
# =========================
yaml_path = os.path.join(OUTPUT_DIR, "data.yaml")
with open(yaml_path, "w", encoding="utf-8") as f:
    f.write(f"path: {OUTPUT_DIR}\n")
    f.write("train: images/train\n")
    f.write("val: images/val\n\n")
    f.write("names:\n")
    for i, name in enumerate(YOLO_CLASS_NAMES):
        f.write(f"  {i}: {name}\n")

print("완료")
print("YOLO 데이터셋 경로:", OUTPUT_DIR)
print("data.yaml:", yaml_path)

총 이미지 수: 797
총 annotation 수: 8270
categories: [{'id': 0, 'name': 'tomatoes', 'supercategory': 'none'}, {'id': 1, 'name': 'ripe', 'supercategory': 'tomatoes'}, {'id': 2, 'name': 'unripe', 'supercategory': 'tomatoes'}]
사용할 이미지 수: 797
train: 638장
val: 159장
완료
YOLO 데이터셋 경로: C:\Users\nva_kist\Desktop\minsun\KIST\Tomato_detect\yolo_dataset
data.yaml: C:\Users\nva_kist\Desktop\minsun\KIST\Tomato_detect\yolo_dataset\data.yaml


#### Yolo26

In [3]:
pip install ultralytics

  Using cached ultralytics-8.4.23-py3-none-any.whl.metadata (39 kB)
  Using cached polars-1.39.0-py3-none-any.whl.metadata (10 kB)
  Using cached ultralytics_thop-2.0.18-py3-none-any.whl.metadata (14 kB)
  Using cached polars_runtime_32-1.39.0-cp310-abi3-win_amd64.whl.metadata (1.5 kB)
Using cached ultralytics-8.4.23-py3-none-any.whl (1.2 MB)
Using cached polars-1.39.0-py3-none-any.whl (823 kB)
Using cached polars_runtime_32-1.39.0-cp310-abi3-win_amd64.whl (47.0 MB)
Using cached ultralytics_thop-2.0.18-py3-none-any.whl (28 kB)

   ---------------------------------------- 0/4 [polars-runtime-32]
   ---------------------------------------- 0/4 [polars-runtime-32]
   ---------- ----------------------------- 1/4 [polars]
   ---------- ----------------------------- 1/4 [polars]
   ---------- ----------------------------- 1/4 [polars]
   ---------- ----------------------------- 1/4 [polars]
   ------------------------------ --------- 3/4 [ultralytics]
   ------------------------------ ------

In [8]:
!nvidia-smi

Tue Mar 17 16:51:07 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 591.44                 Driver Version: 591.44         CUDA Version: 13.1     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                  Driver-Model | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  NVIDIA GeForce RTX 4080 ...  WDDM  |   00000000:01:00.0  On |                  N/A |
|  0%   46C    P8             17W /  320W |    2910MiB /  16376MiB |      1%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+-----

In [9]:
pip uninstall torch torchvision torchaudio -y

Found existing installation: torch 2.10.0
Uninstalling torch-2.10.0:
  Successfully uninstalled torch-2.10.0
Found existing installation: torchvision 0.25.0
Uninstalling torchvision-0.25.0:
  Successfully uninstalled torchvision-0.25.0
Note: you may need to restart the kernel to use updated packages.


You can safely remove it manually.
You can safely remove it manually.


In [10]:
pip install torch torchvision torchaudio --index-url https://download.pytorch.org/whl/cu124

Looking in indexes: https://download.pytorch.org/whl/cu124
  Using cached sympy-1.13.1-py3-none-any.whl.metadata (12 kB)
   ---------------------------------------- 0.0/2.5 GB ? eta -:--:--
   ---------------------------------------- 0.0/2.5 GB 40.0 MB/s eta 0:01:04
   ---------------------------------------- 0.0/2.5 GB 41.3 MB/s eta 0:01:01
   ---------------------------------------- 0.0/2.5 GB 42.1 MB/s eta 0:01:00
    --------------------------------------- 0.0/2.5 GB 41.8 MB/s eta 0:01:00
    --------------------------------------- 0.0/2.5 GB 42.0 MB/s eta 0:01:00
    --------------------------------------- 0.1/2.5 GB 41.9 MB/s eta 0:01:00
    --------------------------------------- 0.1/2.5 GB 41.9 MB/s eta 0:00:59
   - -------------------------------------- 0.1/2.5 GB 42.2 MB/s eta 0:00:59
   - -------------------------------------- 0.1/2.5 GB 42.2 MB/s eta 0:00:59
   - -------------------------------------- 0.1/2.5 GB 42.3 MB/s eta 0:00:58
   - -----------------------------------

ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
lama-cleaner 1.2.5 requires transformers==4.27.4, but you have transformers 5.2.0 which is incompatible.


In [13]:
import torch
print(torch.__version__)
print("cuda available:", torch.cuda.is_available())
print("cuda device count:", torch.cuda.device_count())
print("torch cuda version:", torch.version.cuda)

2.6.0+cu124
cuda available: True
cuda device count: 1
torch cuda version: 12.4


In [ ]:
from ultralytics import YOLO

# Load a pretrained YOLO26n model
model = YOLO(r"C:\Users\nva_kist\Desktop\minsun\KIST\Tomato_detect\yolo26n.pt")

# Train the model on the COCO8 dataset for 100 epochs
train_results = model.train(
    data=r"C:\Users\nva_kist\Desktop\minsun\KIST\Tomato_detect\yolo_dataset\data.yaml",  # Path to dataset configuration file
    epochs=100,  # Number of training epochs
    imgsz=640,  # Image size for training
    device="0",  # Device to run on (e.g., 'cpu', 0, [0,1,2,3])
)

# Evaluate the model's performance on the validation set
metrics = model.val()

# Export the model to ONNX format for deployment
path = model.export(format="onnx")  # Returns the path to the exported model

Ultralytics 8.4.23  Python-3.11.0 torch-2.6.0+cu124 CUDA:0 (NVIDIA GeForce RTX 4080 SUPER, 16376MiB)
engine\trainer: agnostic_nms=False, amp=True, angle=1.0, augment=False, auto_augment=randaugment, batch=16, bgr=0.0, box=7.5, cache=False, cfg=None, classes=None, close_mosaic=10, cls=0.5, compile=False, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=False, cutmix=0.0, data=C:\Users\nva_kist\Desktop\minsun\KIST\Tomato_detect\yolo_dataset\data.yaml, degrees=0.0, deterministic=True, device=0, dfl=1.5, dnn=False, dropout=0.0, dynamic=False, embed=None, end2end=None, epochs=100, erasing=0.4, exist_ok=False, fliplr=0.5, flipud=0.0, format=torchscript, fraction=1.0, freeze=None, half=False, hsv_h=0.015, hsv_s=0.7, hsv_v=0.4, imgsz=640, int8=False, iou=0.7, keras=False, kobj=1.0, line_width=None, lr0=0.01, lrf=0.01, mask_ratio=4, max_det=300, mixup=0.0, mode=train, model=C:\Users\nva_kist\Desktop\minsun\KIST\Tomato_detect\yolo26n.pt, momentum=0.937, mosaic=1.0, multi_scale=0.0, name=t

FileNotFoundError: path/to/image.jpg does not exist

### Add Public dataset

#### TomatOD json 변환


In [4]:
import json

def convert_tomato_classes(input_json, output_json):
    with open(input_json, "r", encoding="utf-8") as f:
        data = json.load(f)

    # 기존 기준
    # 1: unripe
    # 2: semi-ripe
    # 3: fully-ripe
    #
    # 변경 후
    # 0: ripe
    # 1: unripe
    #
    # 매핑:
    # fully-ripe(3) -> ripe(0)
    # unripe(1), semi-ripe(2) -> unripe(1)

    for ann in data["annotations"]:
        old_id = ann["category_id"]

        if old_id == 3:         # fully-ripe -> ripe
            ann["category_id"] = 0
        elif old_id in [1, 2]:  # unripe, semi-ripe -> unripe
            ann["category_id"] = 1
        else:
            raise ValueError(f"알 수 없는 category_id가 있습니다: {old_id}")

    # categories 재구성
    data["categories"] = [
        {
            "supercategory": "tomato",
            "id": 0,
            "name": "ripe"
        },
        {
            "supercategory": "tomato",
            "id": 1,
            "name": "unripe"
        }
    ]

    with open(output_json, "w", encoding="utf-8") as f:
        json.dump(data, f, indent=4, ensure_ascii=False)

    print(f"변환 완료: {output_json}")


# 예시 실행
convert_tomato_classes(r"C:\Users\nva_kist\Desktop\minsun\KIST\Tomato_detect\Public_tomato\TomatOD\annotations\train.json", r"C:\Users\nva_kist\Desktop\minsun\KIST\Tomato_detect\Public_tomato\TomatOD\annotations\train_od.json")
convert_tomato_classes(r"C:\Users\nva_kist\Desktop\minsun\KIST\Tomato_detect\Public_tomato\TomatOD\annotations\test.json", r"C:\Users\nva_kist\Desktop\minsun\KIST\Tomato_detect\Public_tomato\TomatOD\annotations\test_od.json")

변환 완료: C:\Users\nva_kist\Desktop\minsun\KIST\Tomato_detect\Public_tomato\TomatOD\annotations\train_od.json
변환 완료: C:\Users\nva_kist\Desktop\minsun\KIST\Tomato_detect\Public_tomato\TomatOD\annotations\test_od.json


#### LaboroTomato

In [5]:
import json
import os

# =========================
# 경로 설정
# =========================
base_dir = r"C:\Users\nva_kist\Desktop\minsun\KIST\Tomato_detect\Public_tomato\laboro_tomato\annotations"

input_files = [
    os.path.join(base_dir, "train.json"),
    os.path.join(base_dir, "test.json"),
]

# =========================
# 클래스 변환 함수
# =========================
def convert_laboro_tomato_json(input_json, output_json):
    with open(input_json, "r", encoding="utf-8") as f:
        data = json.load(f)

    # 기존 class id
    # 1: b_fully_ripened -> ripe(0)
    # 2: b_half_ripened  -> unripe(1)
    # 3: b_green         -> unripe(1)
    # 4: l_fully_ripened -> ripe(0)
    # 5: l_half_ripened  -> unripe(1)
    # 6: l_green         -> unripe(1)

    for ann in data["annotations"]:
        old_id = ann["category_id"]

        if old_id in [1, 4]:
            ann["category_id"] = 0   # ripe
        elif old_id in [2, 3, 5, 6]:
            ann["category_id"] = 1   # unripe
        else:
            raise ValueError(f"알 수 없는 category_id 발견: {old_id}")

    # categories 재구성
    data["categories"] = [
        {
            "id": 0,
            "name": "ripe",
            "supercategory": "type"
        },
        {
            "id": 1,
            "name": "unripe",
            "supercategory": "type"
        }
    ]

    with open(output_json, "w", encoding="utf-8") as f:
        json.dump(data, f, indent=2, ensure_ascii=False)

    print(f"변환 완료: {output_json}")


# =========================
# 실행
# =========================
for file_path in input_files:
    file_name = os.path.basename(file_path)              # train.json / test.json
    name_only = os.path.splitext(file_name)[0]           # train / test
    output_path = os.path.join(base_dir, f"{name_only}_la.json")

    convert_laboro_tomato_json(file_path, output_path)

변환 완료: C:\Users\nva_kist\Desktop\minsun\KIST\Tomato_detect\Public_tomato\laboro_tomato\annotations\train_la.json
변환 완료: C:\Users\nva_kist\Desktop\minsun\KIST\Tomato_detect\Public_tomato\laboro_tomato\annotations\test_la.json


#### tomatOD + Laboro

In [7]:
import json
import os

# =========================
# 경로 설정
# =========================
la_path = r"C:\Users\nva_kist\Desktop\minsun\KIST\Tomato_detect\Public_tomato\laboro_tomato\annotations\test_la.json"
od_path = r"C:\Users\nva_kist\Desktop\minsun\KIST\Tomato_detect\Public_tomato\TomatOD\annotations\test_od.json"
output_path = r"C:\Users\nva_kist\Desktop\minsun\KIST\Tomato_detect\Public_tomato\test_public.json"


def load_json(path):
    with open(path, "r", encoding="utf-8") as f:
        return json.load(f)


def merge_coco_json(la_data, od_data):
    merged = {}

    # =========================
    # info / licenses / categories
    # =========================
    merged["info"] = {
        "description": "Merged public tomato dataset (Laboro + TomatOD)",
        "version": "1.0"
    }

    merged["licenses"] = la_data.get("licenses", od_data.get("licenses", []))

    # categories는 두 파일이 이미 같은 구조라고 가정
    merged["categories"] = [
        {"id": 0, "name": "ripe", "supercategory": "tomato"},
        {"id": 1, "name": "unripe", "supercategory": "tomato"},
    ]

    merged["images"] = []
    merged["annotations"] = []

    # =========================
    # 1) laboro images/annotations 추가
    # =========================
    new_image_id = 0
    new_ann_id = 0

    la_image_id_map = {}

    for img in la_data["images"]:
        old_id = img["id"]
        new_img = img.copy()
        new_img["id"] = new_image_id
        merged["images"].append(new_img)
        la_image_id_map[old_id] = new_image_id
        new_image_id += 1

    for ann in la_data["annotations"]:
        new_ann = ann.copy()
        new_ann["id"] = new_ann_id
        new_ann["image_id"] = la_image_id_map[ann["image_id"]]
        merged["annotations"].append(new_ann)
        new_ann_id += 1

    # =========================
    # 2) TomatOD images/annotations 추가
    # =========================
    od_image_id_map = {}

    for img in od_data["images"]:
        old_id = img["id"]
        new_img = img.copy()
        new_img["id"] = new_image_id
        merged["images"].append(new_img)
        od_image_id_map[old_id] = new_image_id
        new_image_id += 1

    for ann in od_data["annotations"]:
        new_ann = ann.copy()
        new_ann["id"] = new_ann_id
        new_ann["image_id"] = od_image_id_map[ann["image_id"]]
        merged["annotations"].append(new_ann)
        new_ann_id += 1

    return merged


# =========================
# 실행
# =========================
la_data = load_json(la_path)
od_data = load_json(od_path)

merged_data = merge_coco_json(la_data, od_data)

with open(output_path, "w", encoding="utf-8") as f:
    json.dump(merged_data, f, indent=2, ensure_ascii=False)

print(f"병합 완료: {output_path}")
print(f"총 images 수: {len(merged_data['images'])}")
print(f"총 annotations 수: {len(merged_data['annotations'])}")

병합 완료: C:\Users\nva_kist\Desktop\minsun\KIST\Tomato_detect\Public_tomato\test_public.json
총 images 수: 216
총 annotations 수: 2464


#### Yolo에 맞는 형식으로 변환


In [12]:
import json
import os
import shutil

# =========================
# 원본 경로
# =========================
SRC_BASE = r"C:\Users\nva_kist\Desktop\minsun\KIST\Tomato_detect\Public_tomato"

TRAIN_JSON = os.path.join(SRC_BASE, "annotations", "train_public.json")
TEST_JSON = os.path.join(SRC_BASE, "annotations", "test_public.json")

TRAIN_IMG_DIR = os.path.join(SRC_BASE, "train")
TEST_IMG_DIR = os.path.join(SRC_BASE, "test")

# =========================
# 새 YOLO 데이터셋 경로
# =========================
DST_BASE = r"C:\Users\nva_kist\Desktop\minsun\KIST\Tomato_detect\Public_yolo"

DST_TRAIN_IMG_DIR = os.path.join(DST_BASE, "images", "train")
DST_VAL_IMG_DIR = os.path.join(DST_BASE, "images", "val")

DST_TRAIN_LABEL_DIR = os.path.join(DST_BASE, "labels", "train")
DST_VAL_LABEL_DIR = os.path.join(DST_BASE, "labels", "val")

YAML_PATH = os.path.join(DST_BASE, "data.yaml")

CLASS_NAMES = ["ripe", "unripe"]   # 0: ripe, 1: unripe

# =========================
# 폴더 생성
# =========================
os.makedirs(DST_TRAIN_IMG_DIR, exist_ok=True)
os.makedirs(DST_VAL_IMG_DIR, exist_ok=True)
os.makedirs(DST_TRAIN_LABEL_DIR, exist_ok=True)
os.makedirs(DST_VAL_LABEL_DIR, exist_ok=True)


def copy_images_from_coco(json_path, src_img_dir, dst_img_dir):
    with open(json_path, "r", encoding="utf-8") as f:
        data = json.load(f)

    copied = 0
    missing = 0

    for img in data["images"]:
        file_name = img["file_name"]
        src_path = os.path.join(src_img_dir, file_name)
        dst_path = os.path.join(dst_img_dir, file_name)

        if os.path.exists(src_path):
            shutil.copy2(src_path, dst_path)
            copied += 1
        else:
            print(f"[경고] 이미지 없음: {src_path}")
            missing += 1

    print(f"[복사 완료] {json_path}")
    print(f"  copied: {copied}, missing: {missing}")


def coco_to_yolo_txt(json_path, dst_img_dir, dst_label_dir):
    with open(json_path, "r", encoding="utf-8") as f:
        data = json.load(f)

    images = data["images"]
    annotations = data["annotations"]

    # image_id -> image info
    image_info_map = {img["id"]: img for img in images}

    # image_id -> annotations list
    ann_map = {}
    for ann in annotations:
        image_id = ann["image_id"]
        ann_map.setdefault(image_id, []).append(ann)

    created = 0
    skipped = 0

    for img in images:
        image_id = img["id"]
        file_name = img["file_name"]
        img_w = img["width"]
        img_h = img["height"]

        image_path = os.path.join(dst_img_dir, file_name)
        txt_name = os.path.splitext(file_name)[0] + ".txt"
        txt_path = os.path.join(dst_label_dir, txt_name)

        if not os.path.exists(image_path):
            skipped += 1
            continue

        yolo_lines = []

        for ann in ann_map.get(image_id, []):
            category_id = ann["category_id"]   # 0 or 1
            x, y, w, h = ann["bbox"]           # COCO: [x, y, w, h]

            if w <= 0 or h <= 0:
                continue

            x_center = (x + w / 2) / img_w
            y_center = (y + h / 2) / img_h
            w_norm = w / img_w
            h_norm = h / img_h

            line = f"{category_id} {x_center:.6f} {y_center:.6f} {w_norm:.6f} {h_norm:.6f}"
            yolo_lines.append(line)

        with open(txt_path, "w", encoding="utf-8") as f:
            f.write("\n".join(yolo_lines))

        created += 1

    print(f"[라벨 생성 완료] {json_path}")
    print(f"  created: {created}, skipped(no image): {skipped}")


def create_data_yaml(yaml_path, base_dir, class_names):
    yaml_text = f"""path: {base_dir}
train: images/train
val: images/val

names:
  0: {class_names[0]}
  1: {class_names[1]}
"""
    with open(yaml_path, "w", encoding="utf-8") as f:
        f.write(yaml_text)

    print(f"[data.yaml 생성 완료] {yaml_path}")


# =========================
# 실행
# =========================
# 1) 이미지 복사
copy_images_from_coco(TRAIN_JSON, TRAIN_IMG_DIR, DST_TRAIN_IMG_DIR)
copy_images_from_coco(TEST_JSON, TEST_IMG_DIR, DST_VAL_IMG_DIR)

# 2) YOLO txt 생성
coco_to_yolo_txt(TRAIN_JSON, DST_TRAIN_IMG_DIR, DST_TRAIN_LABEL_DIR)
coco_to_yolo_txt(TEST_JSON, DST_VAL_IMG_DIR, DST_VAL_LABEL_DIR)

# 3) data.yaml 생성
create_data_yaml(YAML_PATH, DST_BASE, CLASS_NAMES)

print("\n모든 작업 완료")
print(f"YOLO 데이터셋 위치: {DST_BASE}")

[복사 완료] C:\Users\nva_kist\Desktop\minsun\KIST\Tomato_detect\Public_tomato\annotations\train_public.json
  copied: 865, missing: 0
[복사 완료] C:\Users\nva_kist\Desktop\minsun\KIST\Tomato_detect\Public_tomato\annotations\test_public.json
  copied: 216, missing: 0
[라벨 생성 완료] C:\Users\nva_kist\Desktop\minsun\KIST\Tomato_detect\Public_tomato\annotations\train_public.json
  created: 865, skipped(no image): 0
[라벨 생성 완료] C:\Users\nva_kist\Desktop\minsun\KIST\Tomato_detect\Public_tomato\annotations\test_public.json
  created: 216, skipped(no image): 0
[data.yaml 생성 완료] C:\Users\nva_kist\Desktop\minsun\KIST\Tomato_detect\Public_yolo\data.yaml

모든 작업 완료
YOLO 데이터셋 위치: C:\Users\nva_kist\Desktop\minsun\KIST\Tomato_detect\Public_yolo


In [14]:
# yolo학습 
from ultralytics import YOLO

# Load a pretrained YOLO26n model
model = YOLO(r"C:\Users\nva_kist\Desktop\minsun\KIST\Tomato_detect\yolo26n.pt")

# Train the model on the public tomato YOLO dataset
train_results = model.train(
    data=r"C:\Users\nva_kist\Desktop\minsun\KIST\Tomato_detect\Public_yolo\data.yaml",
    epochs=100,
    imgsz=640,
    device="0",
)

# Evaluate the model's performance on the validation set
metrics = model.val()

# Export the model to ONNX format for deployment
path = model.export(format="onnx")

Ultralytics 8.4.23  Python-3.11.0 torch-2.6.0+cu124 CUDA:0 (NVIDIA GeForce RTX 4080 SUPER, 16376MiB)
engine\trainer: agnostic_nms=False, amp=True, angle=1.0, augment=False, auto_augment=randaugment, batch=16, bgr=0.0, box=7.5, cache=False, cfg=None, classes=None, close_mosaic=10, cls=0.5, compile=False, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=False, cutmix=0.0, data=C:\Users\nva_kist\Desktop\minsun\KIST\Tomato_detect\Public_yolo\data.yaml, degrees=0.0, deterministic=True, device=0, dfl=1.5, dnn=False, dropout=0.0, dynamic=False, embed=None, end2end=None, epochs=100, erasing=0.4, exist_ok=False, fliplr=0.5, flipud=0.0, format=torchscript, fraction=1.0, freeze=None, half=False, hsv_h=0.015, hsv_s=0.7, hsv_v=0.4, imgsz=640, int8=False, iou=0.7, keras=False, kobj=1.0, line_width=None, lr0=0.01, lrf=0.01, mask_ratio=4, max_det=300, mixup=0.0, mode=train, model=C:\Users\nva_kist\Desktop\minsun\KIST\Tomato_detect\yolo26n.pt, momentum=0.937, mosaic=1.0, multi_scale=0.0, name=tr

ModuleNotFoundError: No module named 'onnx'

In [16]:
pip install onnx

  Using cached protobuf-7.34.0-cp310-abi3-win_amd64.whl.metadata (595 bytes)
  Using cached ml_dtypes-0.5.4-cp311-cp311-win_amd64.whl.metadata (9.2 kB)
   ---------------------------------------- 0.0/16.4 MB ? eta -:--:--
   ---------------- ----------------------- 6.8/16.4 MB 35.0 MB/s eta 0:00:01
   ----------------------------------- ---- 14.4/16.4 MB 36.3 MB/s eta 0:00:01
   ---------------------------------------- 16.4/16.4 MB 33.3 MB/s  0:00:00
Using cached ml_dtypes-0.5.4-cp311-cp311-win_amd64.whl (210 kB)
Using cached protobuf-7.34.0-cp310-abi3-win_amd64.whl (437 kB)

   ---------------------------------------- 0/3 [protobuf]
   -------------------------- ------------- 2/3 [onnx]
   -------------------------- ------------- 2/3 [onnx]
   -------------------------- ------------- 2/3 [onnx]
   -------------------------- ------------- 2/3 [onnx]
   -------------------------- ------------- 2/3 [onnx]
   -------------------------- ------------- 2/3 [onnx]
   -------------------------

In [1]:
# private 으로만 학습 

# =========================
# 경로 설정
# =========================
MODEL_PATH = r"C:\Users\nva_kist\Desktop\minsun\KIST\Tomato_detect\yolo26s.pt"
DATA_PATH = r"C:\Users\nva_kist\Desktop\minsun\KIST\Tomato_detect\Data\private_dataset\dataset.yaml"

RESULT_ROOT = r"C:\Users\nva_kist\Desktop\minsun\KIST\Tomato_detect\trained_private"
RUN_NAME = "yolo26n_640"

# 최종 별도 보관 경로
FINAL_SAVE_DIR = r"C:\Users\nva_kist\Desktop\minsun\KIST\Tomato_detect\final_weights"
FINAL_BEST_PT = os.path.join(FINAL_SAVE_DIR, "private_yolo26n_best.pt")
FINAL_LAST_PT = os.path.join(FINAL_SAVE_DIR, "private_yolo26n_last.pt")
FINAL_ONNX = os.path.join(FINAL_SAVE_DIR, "private_yolo26n_best.onnx")

os.makedirs(FINAL_SAVE_DIR, exist_ok=True)

# =========================
# pretrained model 불러오기
# =========================
model = YOLO(MODEL_PATH)

# =========================
# 학습
# =========================
train_results = model.train(
    data=DATA_PATH,
    epochs=100,
    imgsz=640,
    device=0,
    hsv_h=0,
    hsv_s=0,
    hsv_v=0,
    degrees=3.0,
    translate=0.08,
    scale=0.2,
    fliplr=0.5,
    mosaic=0.3,
    mixup=0.0,
    project=RESULT_ROOT,
    name=RUN_NAME
)

# =========================
# 저장 폴더 확인
# =========================
save_dir = str(train_results.save_dir)
print("학습 결과 폴더:", save_dir)

best_pt_path = os.path.join(save_dir, "weights", "best.pt")
last_pt_path = os.path.join(save_dir, "weights", "last.pt")

print("best.pt 원본 경로:", best_pt_path)
print("last.pt 원본 경로:", last_pt_path)

# =========================
# validation
# =========================
metrics = model.val(data=DATA_PATH)
print("val metrics:", metrics)

# =========================
# best.pt 정보 확인
# =========================
best_model = YOLO(best_pt_path)
best_model.info(detailed=False, imgsz=640)

# =========================
# best.pt / last.pt 별도 경로로 복사
# =========================
if os.path.exists(best_pt_path):
    shutil.copy2(best_pt_path, FINAL_BEST_PT)
    print("best.pt 복사 완료:", FINAL_BEST_PT)

if os.path.exists(last_pt_path):
    shutil.copy2(last_pt_path, FINAL_LAST_PT)
    print("last.pt 복사 완료:", FINAL_LAST_PT)

New https://pypi.org/project/ultralytics/8.4.32 available  Update with 'pip install -U ultralytics'
Ultralytics 8.4.23  Python-3.11.0 torch-2.6.0+cu124 CUDA:0 (NVIDIA GeForce RTX 4080 SUPER, 16376MiB)
engine\trainer: agnostic_nms=False, amp=True, angle=1.0, augment=False, auto_augment=randaugment, batch=16, bgr=0.0, box=7.5, cache=False, cfg=None, classes=None, close_mosaic=10, cls=0.5, compile=False, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=False, cutmix=0.0, data=C:\Users\nva_kist\Desktop\minsun\KIST\Tomato_detect\Data\private_dataset\dataset.yaml, degrees=3.0, deterministic=True, device=0, dfl=1.5, dnn=False, dropout=0.0, dynamic=False, embed=None, end2end=None, epochs=100, erasing=0.4, exist_ok=False, fliplr=0.5, flipud=0.0, format=torchscript, fraction=1.0, freeze=None, half=False, hsv_h=0, hsv_s=0, hsv_v=0, imgsz=640, int8=False, iou=0.7, keras=False, kobj=1.0, line_width=None, lr0=0.01, lrf=0.01, mask_ratio=4, max_det=300, mixup=0.0, mode=train, model=C:\Users\nva

#### Merge된 데이터

In [6]:
from ultralytics import YOLO
from ultralytics.utils.benchmarks import benchmark
import os
import shutil

# =========================
# 경로 설정
# =========================
MODEL_PATH = r"C:\Users\nva_kist\Desktop\minsun\KIST\Tomato_detect\yolo26s.pt"
DATA_PATH = r"C:\Users\nva_kist\Desktop\minsun\KIST\Tomato_detect\Data\Merge\Merge.yaml"

RESULT_ROOT = r"C:\Users\nva_kist\Desktop\minsun\KIST\Tomato_detect\trained_merge"
RUN_NAME = "yolo26s_640"

# 최종 별도 보관 경로
FINAL_SAVE_DIR = r"C:\Users\nva_kist\Desktop\minsun\KIST\Tomato_detect\final_weights"
FINAL_BEST_PT = os.path.join(FINAL_SAVE_DIR, "merge_yolo26s_best.pt")
FINAL_LAST_PT = os.path.join(FINAL_SAVE_DIR, "merge_yolo26s_last.pt")
FINAL_ONNX = os.path.join(FINAL_SAVE_DIR, "merge_yolo26s_best.onnx")

os.makedirs(FINAL_SAVE_DIR, exist_ok=True)

# =========================
# pretrained model 불러오기
# =========================
model = YOLO(MODEL_PATH)

# =========================
# 학습
# =========================
train_results = model.train(
    data=DATA_PATH,
    epochs=100,
    imgsz=640,
    device=0,
    hsv_h=0.0,
    hsv_s=0.2,
    hsv_v=0.3,
    project=RESULT_ROOT,
    name=RUN_NAME
)

# =========================
# 저장 폴더 확인
# =========================
save_dir = str(train_results.save_dir)
print("학습 결과 폴더:", save_dir)

best_pt_path = os.path.join(save_dir, "weights", "best.pt")
last_pt_path = os.path.join(save_dir, "weights", "last.pt")

print("best.pt 원본 경로:", best_pt_path)
print("last.pt 원본 경로:", last_pt_path)

# =========================
# validation
# =========================
metrics = model.val(data=DATA_PATH)
print("val metrics:", metrics)

# =========================
# best.pt 정보 확인
# =========================
best_model = YOLO(best_pt_path)
best_model.info(detailed=False, imgsz=640)

Ultralytics 8.4.23  Python-3.11.0 torch-2.6.0+cu124 CUDA:0 (NVIDIA GeForce RTX 4080 SUPER, 16376MiB)
engine\trainer: agnostic_nms=False, amp=True, angle=1.0, augment=False, auto_augment=randaugment, batch=16, bgr=0.0, box=7.5, cache=False, cfg=None, classes=None, close_mosaic=10, cls=0.5, compile=False, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=False, cutmix=0.0, data=C:\Users\nva_kist\Desktop\minsun\KIST\Tomato_detect\Data\Merge\Merge.yaml, degrees=0.0, deterministic=True, device=0, dfl=1.5, dnn=False, dropout=0.0, dynamic=False, embed=None, end2end=None, epochs=100, erasing=0.4, exist_ok=False, fliplr=0.5, flipud=0.0, format=torchscript, fraction=1.0, freeze=None, half=False, hsv_h=0.0, hsv_s=0.2, hsv_v=0.3, imgsz=640, int8=False, iou=0.7, keras=False, kobj=1.0, line_width=None, lr0=0.01, lrf=0.01, mask_ratio=4, max_det=300, mixup=0.0, mode=train, model=C:\Users\nva_kist\Desktop\minsun\KIST\Tomato_detect\yolo26s.pt, momentum=0.937, mosaic=1.0, multi_scale=0.0, name=yolo

(260, 9949412, 0, 22.503014399999998)

### fine tuning

In [1]:
from ultralytics import YOLO
from ultralytics.utils.benchmarks import benchmark
import os
import shutil

In [ ]:
# =========================
# 경로 설정
# =========================
MODEL_PATH = r"C:\Users\nva_kist\Desktop\minsun\KIST\Tomato_detect\trained_merge\yolo26n_640\weights\best.pt"
DATA_PATH = r"C:\Users\nva_kist\Desktop\minsun\KIST\Tomato_detect\Data\custom_tomato_data_yolo\dataset.yaml"

RESULT_ROOT = r"C:\Users\nva_kist\Desktop\minsun\KIST\Tomato_detect\trained_finetune"
RUN_NAME = "yolo26n_640_finetune"

# 최종 별도 보관 경로
FINAL_SAVE_DIR = r"C:\Users\nva_kist\Desktop\minsun\KIST\Tomato_detect\final_weights"
FINAL_BEST_PT = os.path.join(FINAL_SAVE_DIR, "fine_yolo26n_best.pt")
FINAL_LAST_PT = os.path.join(FINAL_SAVE_DIR, "fine_yolo26n_last.pt")
FINAL_ONNX = os.path.join(FINAL_SAVE_DIR, "fine_yolo26n_best.onnx")

os.makedirs(FINAL_SAVE_DIR, exist_ok=True)

# =========================
# pretrained model 불러오기
# =========================
model = YOLO(MODEL_PATH)

# =========================
# 학습
# =========================
train_results = model.train(
    data=DATA_PATH,
    epochs=100,
    imgsz=640,
    device=0,
    hsv_h=0.0,
    hsv_s=0.2,
    hsv_v=0.3,
    project=RESULT_ROOT,
    name=RUN_NAME
)

# =========================
# 저장 폴더 확인
# =========================
save_dir = str(train_results.save_dir)
print("학습 결과 폴더:", save_dir)

best_pt_path = os.path.join(save_dir, "weights", "best.pt")
last_pt_path = os.path.join(save_dir, "weights", "last.pt")

print("best.pt 원본 경로:", best_pt_path)
print("last.pt 원본 경로:", last_pt_path)

# =========================
# validation
# =========================
metrics = model.val(data=DATA_PATH)
print("val metrics:", metrics)

# =========================
# best.pt 정보 확인
# =========================
best_model = YOLO(best_pt_path)
best_model.info(detailed=False, imgsz=640)

New https://pypi.org/project/ultralytics/8.4.24 available  Update with 'pip install -U ultralytics'
Ultralytics 8.4.23  Python-3.11.0 torch-2.6.0+cu124 CUDA:0 (NVIDIA GeForce RTX 4080 SUPER, 16376MiB)
engine\trainer: agnostic_nms=False, amp=True, angle=1.0, augment=False, auto_augment=randaugment, batch=16, bgr=0.0, box=7.5, cache=False, cfg=None, classes=None, close_mosaic=10, cls=0.5, compile=False, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=False, cutmix=0.0, data=C:\Users\nva_kist\Desktop\minsun\KIST\Tomato_detect\Data\custom_tomato_data_yolo\dataset.yaml, degrees=0.0, deterministic=True, device=0, dfl=1.5, dnn=False, dropout=0.0, dynamic=False, embed=None, end2end=None, epochs=100, erasing=0.4, exist_ok=False, fliplr=0.5, flipud=0.0, format=torchscript, fraction=1.0, freeze=None, half=False, hsv_h=0.015, hsv_s=0.7, hsv_v=0.4, imgsz=640, int8=False, iou=0.7, keras=False, kobj=1.0, line_width=None, lr0=0.01, lrf=0.01, mask_ratio=4, max_det=300, mixup=0.0, mode=train, mo

(260, 2504580, 0, 5.773926400000001)

In [ ]:
# =========================
# 경로 설정
# =========================
MODEL_PATH = r"C:\Users\nva_kist\Desktop\minsun\KIST\Tomato_detect\trained_merge\yolo26n_640\weights\best.pt"
DATA_PATH = r"C:\Users\nva_kist\Desktop\minsun\KIST\Tomato_detect\Data\custom_tomato_data_yolo\dataset.yaml"

RESULT_ROOT = r"C:\Users\nva_kist\Desktop\minsun\KIST\Tomato_detect\trained_finetune"
RUN_NAME = "yolo26n_640_finetune"

# 최종 별도 보관 경로
FINAL_SAVE_DIR = r"C:\Users\nva_kist\Desktop\minsun\KIST\Tomato_detect\final_weights"
FINAL_BEST_PT = os.path.join(FINAL_SAVE_DIR, "fine_yolo26n_best.pt")
FINAL_LAST_PT = os.path.join(FINAL_SAVE_DIR, "fine_yolo26n_last.pt")
FINAL_ONNX = os.path.join(FINAL_SAVE_DIR, "fine_yolo26n_best.onnx")

os.makedirs(FINAL_SAVE_DIR, exist_ok=True)

# =========================
# pretrained model 불러오기
# =========================
model = YOLO(MODEL_PATH)

# =========================
# 학습
# =========================
train_results = model.train(
    data=DATA_PATH,
    epochs=100,
    imgsz=640,
    device=0,
    hsv_h=0.0,
    hsv_s=0.2,
    hsv_v=0.3,
    project=RESULT_ROOT,
    name=RUN_NAME
)

# =========================
# 저장 폴더 확인
# =========================
save_dir = str(train_results.save_dir)
print("학습 결과 폴더:", save_dir)

best_pt_path = os.path.join(save_dir, "weights", "best.pt")
last_pt_path = os.path.join(save_dir, "weights", "last.pt")

print("best.pt 원본 경로:", best_pt_path)
print("last.pt 원본 경로:", last_pt_path)

# =========================
# validation
# =========================
metrics = model.val(data=DATA_PATH)
print("val metrics:", metrics)

# =========================
# best.pt 정보 확인
# =========================
best_model = YOLO(best_pt_path)
best_model.info(detailed=False, imgsz=640)

In [1]:
import os
import random
import shutil
from pathlib import Path

# =========================
# 설정
# =========================
random.seed(42)

base_dir = Path(r"C:\Users\nva_kist\Desktop\minsun\KIST\Tomato_detect\Data\private_dataset")

train_label_dir = base_dir / "labels" / "train"
val_label_dir   = base_dir / "labels" / "val"

train_image_dir = base_dir / "images" / "train"
val_image_dir   = base_dir / "images" / "val"

val_ratio = 0.2

# =========================
# val 폴더 생성
# =========================
val_label_dir.mkdir(parents=True, exist_ok=True)
val_image_dir.mkdir(parents=True, exist_ok=True)

# =========================
# train 라벨 파일 목록
# =========================
label_files = sorted(train_label_dir.glob("*.txt"))
print(f"train label 개수: {len(label_files)}")

# 숨김/시스템 파일 방지용
label_files = [f for f in label_files if f.is_file()]

# =========================
# 섞고 20% 선택
# =========================
random.shuffle(label_files)
n_val = int(len(label_files) * val_ratio)
val_label_files = label_files[:n_val]

print(f"val로 이동할 개수: {n_val}")

# =========================
# 이미지 확장자 후보
# =========================
img_exts = [".jpg", ".jpeg", ".png", ".bmp", ".tif", ".tiff"]

# =========================
# 이동
# =========================
moved_count = 0
missing_images = []

for label_path in val_label_files:
    stem = label_path.stem

    # 대응 이미지 찾기
    image_path = None
    for ext in img_exts:
        candidate = train_image_dir / f"{stem}{ext}"
        if candidate.exists():
            image_path = candidate
            break

    if image_path is None:
        missing_images.append(stem)
        continue

    # 라벨 이동
    shutil.move(str(label_path), str(val_label_dir / label_path.name))

    # 이미지 이동
    shutil.move(str(image_path), str(val_image_dir / image_path.name))

    moved_count += 1

# =========================
# 결과 출력
# =========================
print(f"\n실제로 val로 이동한 쌍 개수: {moved_count}")

if missing_images:
    print("\n대응 이미지 못 찾은 파일들:")
    for x in missing_images:
        print(x)

print(f"\n최종 train labels 개수: {len(list(train_label_dir.glob('*.txt')))}")
print(f"최종 val labels 개수: {len(list(val_label_dir.glob('*.txt')))}")
print(f"최종 train images 개수: {len([x for x in train_image_dir.iterdir() if x.is_file()])}")
print(f"최종 val images 개수: {len([x for x in val_image_dir.iterdir() if x.is_file()])}")

train label 개수: 1424
val로 이동할 개수: 284

실제로 val로 이동한 쌍 개수: 284

최종 train labels 개수: 1140
최종 val labels 개수: 284
최종 train images 개수: 1140
최종 val images 개수: 284
